In [1]:
# ==============================================================================
# FILE/CELL 2: sensitivity_p.py
# Sensitivity of the threshold estimator to the target tail depth (p)
# ==============================================================================

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import multiprocessing
from udrud_framework import detect_plateau_and_estimate
import matplotlib.pyplot as plt

# 1. Setup Parameters
k_min = 20           # Fixed lower bound
window_size = 5      # Fixed smoothing window
p_test_list = [0.025, 0.050, 0.075, 0.100, 0.125] # 2.5% to 12.5% tail depths

n_cores = multiprocessing.cpu_count()

# 2. Define the parallelized scenario runner
def compute_p_scenario(p_val, N, cum_w_arr, y, w, k_min_val, ws):
    """Calculates k_max dynamically based on p, then runs detection."""
    target_weight = p_val * N

    # Fast ECDF lookup for the new K_max
    k_max_val = np.searchsorted(cum_w_arr, target_weight, side='right')
    k_range_obj = range(k_min_val, max(k_min_val + 1, k_max_val))

    gamma_hat, best_k = detect_plateau_and_estimate(y, w, k_range_obj, window_size=ws)
    return {'p': p_val, 'k_max': k_max_val, 'gamma_hat': gamma_hat, 'k^*': best_k}

# Load your empirical dataset
df = pd.read_csv("2021_2025_disposable_income.csv")

# equivalize income and weight
df["weight"] = df["weight"] * df["size"]
df["income"] = df["income"] / np.sqrt(df["size"])

# 3. Load Data (assuming df is already loaded and equivalized)
results_p = []

for year, group in df.groupby("year"):
    y_val = group["income"].values
    w_val = group["weight"].values

    # Pre-compute the empirical distribution ONCE per year to save execution time
    N = np.sum(w_val)
    support_df = pd.DataFrame({'y': y_val, 'w': w_val}).groupby('y').sum().sort_index()
    cum_w = support_df['w'].cumsum().values

    # Run scenarios in parallel across the p list
    rs = Parallel(n_jobs=n_cores)(
        delayed(compute_p_scenario)(p, N, cum_w, y_val, w_val, k_min, window_size)
        for p in p_test_list
    )

    for res_dict in rs:
        res_dict['year'] = year
        results_p.append(res_dict)

# 4. Results & Diagnostics
df_p_results = pd.DataFrame(results_p)

print(df_p_results)

        p  k_max     gamma_hat   k^*  year
0   0.025    576 -38192.447596   575  2021
1   0.050    989 -38180.703019   988  2021
2   0.075   1370 -38176.149064  1369  2021
3   0.100   1744 -38173.614322  1743  2021
4   0.125   2104 -38172.025736  2103  2021
5   0.025    593  -5855.361334    67  2022
6   0.050   1033  -5853.680013  1032  2022
7   0.075   1438  -5853.611622  1437  2022
8   0.100   1829  -5853.573669  1828  2022
9   0.125   2173  -5853.551353  2172  2022
10  0.025    616 -21093.329652   615  2023
11  0.050   1040 -21090.098056  1039  2023
12  0.075   1472 -21088.715527  1471  2023
13  0.100   1865 -21088.013092  1864  2023
14  0.125   2257 -21087.555739  2256  2023
15  0.025    634 -18493.506457   633  2024
16  0.050   1087 -18489.749215  1086  2024
17  0.075   1523 -18488.241812  1522  2024
18  0.100   1952 -18487.415363  1951  2024
19  0.125   2365 -18486.902816  2364  2024
20  0.025    614  -6102.060326   219  2025
21  0.050   1069  -6102.060326   219  2025
22  0.075  